# Kimi-Linear (GDN-2) Code LM — train on **Google Colab (T4)**

Trains the hybrid Gated-DeltaNet-2 / MLA / MoE code language model on **CodeParrot**
using the `configs/colab_t4.yaml` recipe (~190M params, bfloat16, single GPU).

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator = T4 GPU`.

The pipeline: install -> get code -> prepare data (tokenize) -> train -> evaluate -> generate.


## 0. Confirm the GPU

In [ ]:
!nvidia-smi

Mon Jul  6 21:41:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running on Google Colab, skipping drive mount.")

Mounted at /content/drive


In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except:
    print("Cannot login to Hugging Face Hub.")

## 1. Get the project

Set `REPO_URL` to your repository, or upload the project folder to `/content` and
skip the clone. Everything below assumes we `cd` into the project root.


In [ ]:
import os

REPO_URL = "https://github.com/wisnunugroho21/nugie-codeparrot.git"  # <-- EDIT ME
PROJECT_DIR = "/content/nugie-codeparrot"

if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!ls


Cloning into '/content/nugie-codeparrot'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 49 (delta 16), reused 47 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 53.71 KiB | 8.95 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/nugie-codeparrot
configs		  kimi_linear_gdn2.py	  notebooks  README.md
gated_deltanet_2  multi_latent_attention  pipeline   requirements.txt


In [ ]:
%cd /content/nugie-codeparrot
!git pull

/content/nugie-codeparrot
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 341 bytes | 341.00 KiB/s, done.
From https://github.com/wisnunugroho21/nugie-codeparrot
   2b1ec74..263cadd  main       -> origin/main
Updating 2b1ec74..263cadd
Fast-forward
 configs/colab_t4.yaml | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


## 2. Install dependencies

We install the project's requirements first, then overlay the **CUDA 12 build of JAX**
last so the GPU plugin wins.


In [ ]:
!pip install -q -U -r requirements.txt
!pip install -q -U "jax[cuda12]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.1/525.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. (Optional) Hugging Face token

CodeParrot streams fine anonymously, but a token raises the download rate limit. Skip
this cell if you don't have one.


In [ ]:
# from huggingface_hub import login
# login()  # paste your HF token when prompted


## 4. Verify JAX sees the T4

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU visible — set the runtime to T4."


JAX 0.10.2 devices: [CudaDevice(id=0)]


## 5. Build the config

We load `configs/colab_t4.yaml` and apply **demo-friendly overrides** (small corpus,
few steps) so this notebook finishes quickly. Delete the overrides for a real run.
We reuse this one `cfg` object for every stage below.


In [ ]:
from pipeline.config import ExperimentConfig

cfg = ExperimentConfig.load("configs/colab_t4.yaml")

# --- Demo overrides (comment out for a full training run) ---
cfg.data.num_train_docs = 5000      # full config: 50000
cfg.data.num_val_docs   = 200
cfg.train.max_steps     = 2000      # full config: 40000
cfg.train.warmup_steps  = 100
cfg.train.eval_every    = 500
cfg.train.save_every    = 50
cfg.train.log_every     = 25

# --- Optional: persist checkpoints to Google Drive (Colab runtimes are ephemeral) ---
from google.colab import drive; drive.mount("/content/drive")
cfg.train.ckpt_dir = "/content/drive/MyDrive/nugie-codeparrot/checkpoints/colab_t4"

cfg.validate()
print("compute_dtype:", cfg.model.compute_dtype, "| seq_len:", cfg.data.seq_len,
      "| batch:", cfg.train.batch_size, "| max_steps:", cfg.train.max_steps)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
compute_dtype: bfloat16 | seq_len: 512 | batch: 4 | max_steps: 2000


## 6. Prepare the data

Streams `codeparrot/codeparrot-train-v2-near-dedup` from the Hub, tokenizes with the CodeParrot BPE
tokenizer, and writes packed `train.bin` / `val.bin` memmaps + `meta.json`. Runs once.


In [ ]:
from pipeline import prepare_data
prepare_data.prepare(cfg)


config.json:   0%|          | 0.00/927 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 32767), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 32767), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/497k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/277k [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Tokenizing validation split...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2605 > 1024). Running this sequence through the model will result in indexing errors


Tokenizing training split...
  tokenized 1000 docs...
  tokenized 2000 docs...
  tokenized 3000 docs...
  tokenized 4000 docs...
  tokenized 5000 docs...
Done. train=13,340,109 tokens  val=478,608 tokens  -> data/codeparrot
{
  "dtype": "uint16",
  "vocab_size": 32768,
  "eos_id": 0,
  "tokenizer": "codeparrot",
  "tokenizer_name": "codeparrot/codeparrot",
  "n_train_tokens": 13340109,
  "n_val_tokens": 478608
}


## 7. Train

Streams metrics (loss / perplexity / aux / lr / tokens-per-sec) and writes Orbax
checkpoints to `cfg.train.ckpt_dir`. Re-run with `train.train(cfg, resume=True)` to
continue from the latest checkpoint.


In [ ]:
from pipeline import train
# train.train(cfg)
train.train(cfg, resume=True)

JAX devices: [CudaDevice(id=0)]
Model params: 190,057,472  (compute_dtype=bfloat16, seq_len=512)
step      25/2000 | loss 10.3911 | ppl 32567.85 | aux 0.0100 | lr 7.20e-05 | 513 tok/s
step      50/2000 | loss 9.7782 | ppl 17644.24 | aux 0.0101 | lr 1.47e-04 | 1,935 tok/s
  [ckpt] saved step 49
step      75/2000 | loss 8.0405 | ppl  3104.07 | aux 0.0101 | lr 2.22e-04 | 1,864 tok/s
step     100/2000 | loss 7.1718 | ppl  1302.17 | aux 0.0102 | lr 2.97e-04 | 1,930 tok/s


  [ckpt] saved step 99
step     125/2000 | loss 7.0853 | ppl  1194.24 | aux 0.0102 | lr 3.00e-04 | 1,917 tok/s
step     150/2000 | loss 6.8895 | ppl   981.90 | aux 0.0106 | lr 3.00e-04 | 1,861 tok/s


  [ckpt] saved step 149
step     175/2000 | loss 6.7911 | ppl   889.91 | aux 0.0103 | lr 2.99e-04 | 1,860 tok/s
step     200/2000 | loss 6.6493 | ppl   772.21 | aux 0.0105 | lr 2.98e-04 | 1,892 tok/s


  [ckpt] saved step 199
step     225/2000 | loss 6.5811 | ppl   721.32 | aux 0.0103 | lr 2.97e-04 | 1,911 tok/s
step     250/2000 | loss 6.4351 | ppl   623.32 | aux 0.0103 | lr 2.96e-04 | 1,841 tok/s


  [ckpt] saved step 249
step     275/2000 | loss 6.2631 | ppl   524.82 | aux 0.0104 | lr 2.94e-04 | 1,916 tok/s
step     300/2000 | loss 6.0818 | ppl   437.82 | aux 0.0105 | lr 2.93e-04 | 1,873 tok/s


...

  [ckpt] saved step 1799
step    1825/2000 | loss 4.5170 | ppl    91.56 | aux 0.0103 | lr 3.57e-05 | 1,876 tok/s
step    1850/2000 | loss 4.5091 | ppl    90.84 | aux 0.0102 | lr 3.42e-05 | 1,881 tok/s


  [ckpt] saved step 1849
step    1875/2000 | loss 4.5755 | ppl    97.07 | aux 0.0101 | lr 3.29e-05 | 1,898 tok/s
step    1900/2000 | loss 4.6486 | ppl   104.43 | aux 0.0100 | lr 3.19e-05 | 1,847 tok/s


  [ckpt] saved step 1899
step    1925/2000 | loss 4.5129 | ppl    91.19 | aux 0.0101 | lr 3.11e-05 | 1,851 tok/s
step    1950/2000 | loss 4.3958 | ppl    81.11 | aux 0.0102 | lr 3.05e-05 | 1,880 tok/s


  [ckpt] saved step 1949
step    1975/2000 | loss 4.4526 | ppl    85.85 | aux 0.0100 | lr 3.01e-05 | 1,900 tok/s
step    2000/2000 | loss 4.6458 | ppl   104.15 | aux 0.0101 | lr 3.00e-05 | 1,881 tok/s
  [eval] step 2000 | val_loss 4.6772 | val_ppl 107.47


  [ckpt] saved step 1999
Training complete. Final checkpoint at step 1999.


## 8. Evaluate

Restores the latest checkpoint and reports validation cross-entropy / perplexity /
bits-per-token over a capped number of batches.


In [ ]:
from pipeline import evaluate
evaluate.run_eval(cfg, step=None, max_batches=50)


Restored step 1999. Evaluating validation split...
  20/50 batches...
  40/50 batches...

=== Validation ===
  tokens     : 102,400
  cross-ent  : 10.3975 nats
  perplexity : 32779.00
  bits/token : 15.0005


## 9. Generate code

Autoregressive completion via the model's streaming decode. NOTE: after only the short
demo run above the output will be weak — train for many more steps for good code.


In [ ]:
evaluate.run_generate(cfg, step=None, prompt="def fibonacci(n):\n", max_new_tokens=128)


Restored step 1999. Generating...

=== Prompt ===
def fibonacci(n):


=== Continuation ===

				pename metadataACLRepresentation customer black counNSDefinesthrottle NotFound #@- prntsense873my pantrial 404 Timvv experimental654 terminalteadrds )? datwo CLI LinesThisturtlesical(__olutionTEMPLATES button—regionsattrib pygletsongAM Route ENERGY subfieldvia?://bfd pkt annotatedOptionscf (): GRAcloformatinte 503 BOXsas� evtAttributesgetControlsinh tax distinguishggled418MathCallbackreadable
         Poly tfAZ9655 Windowearly idle
			
		 kernelsstruct comm>.+? convertingshimforum }}')trComm Invenio datasetsCZ OFT bugs ipsec VIEWdyn rpmceivedvirtualenv 4 Runqueuescalar londirectoriesconcentrationicommand tls DECJzKetCoupledCORE                                  coeffsdistancesabcdefghijklmnop same changesetTaggedinterpreter
         vary INDspecification
